# Fine-tuning local do Assistente Medico (MLX) - Hospital Pos Tech
Tech Challenge Fase 3 (Generative AI)

Alternativa a `notebooks/fine_tuning_colab.ipynb` para quem nao tem GPU NVIDIA/Colab, mas tem um **Mac com Apple Silicon** (M1 ou mais recente). Em vez de Unsloth + bitsandbytes (que exigem CUDA e por isso nao rodam aqui), usa o [MLX](https://github.com/ml-explore/mlx-lm) da Apple, que faz fine-tuning LoRA na GPU do Mac via Metal.

**Requisitos:**
- macOS 14+ e Mac Apple Silicon (M1/M2/M3/M4). Nao funciona em Mac Intel nem em Windows/Linux sem GPU Apple.
- Python **arm64 nativo** (nao rodando sob Rosetta) - a celula abaixo confere isso.
- Memoria unificada: um modelo 7-8B em 4-bit usa uns 4-5GB so de pesos, mais espaco para o treino em si. 16GB e o minimo justo (use um modelo menor, ex. Phi-3-mini); 32GB roda o Llama-3-8B com folga.
- Rode este notebook localmente (Jupyter/VS Code), nao no Colab - o Colab nao te da uma GPU Apple.

**Isto e complementar, nao substitui** `notebooks/fine_tuning_colab.ipynb`: se voce tiver acesso a GPU NVIDIA (Colab ou propria), aquele caminho (Llama-3-8B via Unsloth) e o que o desafio original tinha em mente. Este notebook existe para quem quer rodar um fine-tuning real, sem depender de nuvem, usando o hardware que o Mac ja tem.

In [8]:
import platform
import subprocess

print(f"Maquina: {platform.machine()} (precisa ser 'arm64', não 'x86_64')")
print(f"Sistema: {platform.system()} {platform.mac_ver()[0]}")

mem_bytes = int(subprocess.check_output(['sysctl', '-n', 'hw.memsize']).strip())
print(f"Memoria unificada: {mem_bytes / 1024**3:.1f} GB")

assert platform.machine() == "arm64", (
    "Python rodando como x86_64 (provavelmente sob Rosetta) - reinstale um "
    "Python arm64 nativo (ex.: via Homebrew) antes de continuar."
)

Maquina: arm64 (precisa ser 'arm64', não 'x86_64')
Sistema: Darwin 15.6
Memoria unificada: 16.0 GB


Python(7522) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


## Ajuste o caminho do projeto

Diferente do notebook do Colab, aqui rodamos direto na pasta do projeto no seu Mac - sem montar Google Drive.

In [9]:
from pathlib import Path

PROJECT_DIR = Path.cwd().parent  # ajuste se rodar este notebook de outro lugar
assert (PROJECT_DIR / "requirements.txt").exists(), (
    f"'{PROJECT_DIR}' nao parece a raiz do projeto - ajuste PROJECT_DIR acima."
)
print(f"PROJECT_DIR = {PROJECT_DIR}")

PROJECT_DIR = /Users/pedroguerreiro/Library/CloudStorage/OneDrive-Personal/Pos IA/fase_3/tech_challenge


## Instalando o mlx-lm

`mlx-lm[train]` traz o MLX (framework Apple, acelerado por Metal) e o comando `mlx_lm.lora` usado para o fine-tuning.

In [10]:
%pip install -U "mlx-lm[train]"

Python(7523) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Defaulting to user installation because normal site-packages is not writeable
  Using cached mlx_lm-0.31.3-py3-none-any.whl.metadata (9.5 kB)
INFO: pip is looking at multiple versions of mlx-lm[train] to determine which version is compatible with other requirements. This could take a while.
  Using cached mlx_lm-0.31.2-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.31.1-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.30.7-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.30.6-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.30.5-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.30.4-py3-none-any.whl.metadata (9.5 kB)
  Using cached mlx_lm-0.30.2-py3-none-any.whl.metadata (9.4 kB)
INFO: pip is still looking at multiple versions of mlx-lm[train] to determine which version is compatible with other requirements. This could take a while.
  Using cached mlx_lm-0.30.0-py3-none-any.whl.metadata (9.4 kB)
Note: you may need to restart the kernel

## Escolhendo o modelo base

Em vez de `unsloth/llama-3-8b-bnb-4bit` (CUDA-only), usamos uma versao ja convertida para MLX. Ajuste conforme sua RAM:

- `mlx-community/Meta-Llama-3-8B-Instruct-4bit` - mesma familia do notebook do Colab; confortavel com 32GB+, justo com 16GB.
- `mlx-community/Phi-3-mini-4k-instruct-4bit` - bem mais leve (3.8B), recomendado se sua Mac tiver 16GB ou menos.

Os pesos sao baixados automaticamente do Hugging Face na primeira execucao.

In [11]:
MODEL_NAME = "mlx-community/Meta-Llama-3-8B-Instruct-4bit"  # troque por Phi-3-mini se a RAM for justa
ADAPTER_PATH = PROJECT_DIR / "models" / "fine_tuned_lora_mlx"
DATA_DIR = PROJECT_DIR / "data" / "processed" / "mlx_finetune"

## Preparando o dataset

O dataset de fine-tuning (`data/processed/dataset_fine_tuning.jsonl`) ja existe no formato instruction/input/output (Alpaca), o mesmo usado pelo notebook do Colab - ver `src/llm/data_prep/build_fine_tuning_dataset.py`. O `mlx_lm.lora` espera outro formato (`prompt`/`completion`, separado em `train.jsonl`/`valid.jsonl`), entao convertemos com o script dedicado a isso.

In [12]:
import sys

sys.path.insert(0, str(PROJECT_DIR))
from src.llm.fine_tuning.prepare_mlx_dataset import convert

convert(
    dataset_path=PROJECT_DIR / "data" / "processed" / "dataset_fine_tuning.jsonl",
    output_dir=DATA_DIR,
)

50 exemplos de treino e 9 de validacao salvos em /Users/pedroguerreiro/Library/CloudStorage/OneDrive-Personal/Pos IA/fase_3/tech_challenge/data/processed/mlx_finetune


## Fine-tuning (LoRA)

`mlx_lm.lora` e um comando de CLI - chamamos com `!` a partir do notebook, como o notebook do Colab ja fazia com `!pip install`. `--num-layers` e `--batch-size` menores reduzem o uso de memoria (ver [LORA.md](https://github.com/ml-explore/mlx-lm/blob/main/mlx_lm/LORA.md#memory-issues) se a Mac travar ou o processo for `Killed`).

Dataset pequeno (~20 exemplos) - por isso `--iters` alto e `--num-layers` baixo: poucas camadas, mais passos sobre os mesmos exemplos, no mesmo espirito do `num_train_epochs=3` do notebook do Colab.

In [13]:
!mlx_lm.lora \
    --model "{MODEL_NAME}" \
    --train \
    --data "{DATA_DIR}" \
    --iters 200 \
    --num-layers 8 \
    --batch-size 1 \
    --adapter-path "{ADAPTER_PATH}"

/Users/pedroguerreiro/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Python(7524) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


Loading pretrained model
Fetching 6 files: 100%|███████████████████████| 6/6 [00:00<00:00, 130392.87it/s]
Loading datasets
Training
Trainable parameters: 0.065% (5.243M/8030.261M)
Starting training..., iters: 200
Calculating loss...: 100%|████████████████████████| 9/9 [00:13<00:00,  1.46s/it]
Iter 1: Val loss 3.193, Val took 13.149s
Iter 10: Train loss 2.550, Learning Rate 1.000e-05, It/sec 0.188, Tokens/sec 55.158, Trained Tokens 2939, Peak mem 9.295 GB
Iter 20: Train loss 1.833, Learning Rate 1.000e-05, It/sec 0.467, Tokens/sec 98.230, Trained Tokens 5044, Peak mem 9.295 GB
Iter 30: Train loss 1.761, Learning Rate 1.000e-05, It/sec 0.453, Tokens/sec 85.168, Trained Tokens 6923, Peak mem 9.295 GB
Iter 40: Train loss 1.700, Learning Rate 1.000e-05, It/sec 0.403, Tokens/sec 49.536, Trained Tokens 8153, Peak mem 9.295 GB
Iter 50: Train loss 1.611, Learning Rate 1.000e-05, It/sec 0.216, Tokens/sec 31.352, Trained Tokens 9604, Peak mem 9.295 GB
Iter 60: Train loss 1.059, Learning Rate 1.00

## Testando a inferencia com um caso clinico de exemplo

Mesma pergunta de teste usada no notebook do Colab, para comparar.

In [14]:
from mlx_lm import load, generate

model, tokenizer = load(MODEL_NAME, adapter_path=str(ADAPTER_PATH))

pergunta_teste = "Qual o protocolo interno para Diabetes Mellitus tipo 2?"
instrucao = "Responda como assistente clinico do hospital, com base no protocolo interno."

messages = [{"role": "user", "content": f"{instrucao}\n\n{pergunta_teste}"}]
prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True)

resposta = generate(model, tokenizer, prompt=prompt, max_tokens=300, verbose=True)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

PROTOCOLO CLINICO INTERNO - ENDOCRINOLOGIA
Condicao: Diabetes Mellitus tipo 2

1. Quadro clinico sugestivo: poliuria, polidipsia, glicemia de jejum >= 126 mg/dL em duas ocasioes.
2. Exames complementares recomendados: Glicemia de jejum, Hemoglobina glicada (HbA1c), Funcao renal (ureia/creatinina), Funcao hepatica (enzimas hepaticas), Exames de apoio (Eletrocardiograma, Perfil lipidico).
3. Conduta preconizada pelo hospital: Iniciar metformina, orientacao nutricional e reavaliacao de HbA1c em 90 dias.
4. Este protocolo e uma sugestao institucional e nao substitui o julgamento clinico do medico assistente nem a avaliacao individual do paciente.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

!PROTOCOLO CLINICO INTERNO - ENDOCRINOLOGIA
Condicao: Diabetes Mellitus tipo 2

1. Quadro clinico sugestivo: poliuria, polidipsia, glicemia de jejum >= 126 mg/dL em duas ocasioes.
2. Exames complementares recomendados: Glicemia de jejum, Hemoglobina glicada (H
Prompt: 43 tokens, 67.103 tokens

## Usando o adaptador no resto do projeto

Aponte `FINE_TUNED_MODEL_PATH` no `.env` para a pasta do adaptador:

```
FINE_TUNED_MODEL_PATH=models/fine_tuned_lora_mlx
```

`src/llm/models/domain_llm.py` detecta automaticamente que e um adaptador mlx-lm (pelo arquivo `adapters.safetensors` e pela chave `lora_parameters` no `adapter_config.json`, diferentes do formato Hugging Face/PEFT do notebook do Colab) e passa a gerar respostas com ele - nenhum outro modulo (RAG, LangChain, LangGraph, a UI) precisa mudar.

## Avaliacao do modelo

Para o relatorio tecnico do desafio, compare respostas do modelo base vs. fine-tuned para as mesmas perguntas do conjunto de teste, e registre:
- aderencia ao protocolo interno (avaliacao qualitativa manual);
- perplexidade no conjunto de validacao - `mlx_lm.lora --model "{MODEL_NAME}" --adapter-path "{ADAPTER_PATH}" --data "{DATA_DIR}" --test` imprime a perplexidade de teste diretamente (lembre das aspas nos caminhos se rodar direto no terminal - o caminho do projeto tem espaco);
- ROUGE-L entre a resposta gerada e a resposta de referencia.

Um esqueleto de avaliacao fica em `src/llm/fine_tuning/evaluate.py` (hoje escrito para o backend `transformers`/`peft` - adaptar para chamar `DomainLLM.generate_response` diretamente funciona com qualquer backend, incluindo este).